In [0]:
%pip install scikit-learn

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
import numpy as np

# Load silver table
df = spark.sql("SELECT * FROM silver_stock_analytics ORDER BY Ticker, Date").toPandas()
print(f"Rows loaded: {len(df)}")

Rows loaded: 63550


In [0]:
from sklearn.ensemble import IsolationForest

# Create features that describe each trading day
anomaly_features = []

for ticker in df["Ticker"].unique():
    stock = df[df["Ticker"] == ticker].sort_values("Date").copy()
    
    # How extreme was today's return compared to this stock's history?
    stock["return_zscore"] = (stock["daily_return_pct"] - stock["daily_return_pct"].rolling(60).mean()) / stock["daily_return_pct"].rolling(60).std()
    
    # How extreme was today's volume compared to recent average?
    stock["volume_zscore"] = (stock["Volume"] - stock["Volume"].rolling(60).mean()) / stock["Volume"].rolling(60).std()
    
    # How big was the daily price swing?
    stock["daily_range_pct"] = (stock["High"] - stock["Low"]) / stock["Close"] * 100
    stock["range_zscore"] = (stock["daily_range_pct"] - stock["daily_range_pct"].rolling(60).mean()) / stock["daily_range_pct"].rolling(60).std()
    
    # Gap between today's open and yesterday's close (overnight surprise)
    stock["overnight_gap"] = ((stock["Open"] - stock["prev_close"]) / stock["prev_close"]) * 100
    
    anomaly_features.append(stock)

anomaly_df = pd.concat(anomaly_features).dropna()

feature_cols_anomaly = ["return_zscore", "volume_zscore", "range_zscore", "overnight_gap", "daily_return_pct"]

print(f"Total samples: {len(anomaly_df)}")
print(f"\nFeature stats:")
print(anomaly_df[feature_cols_anomaly].describe().round(2))

Total samples: 60600

Feature stats:
       return_zscore  volume_zscore  ...  overnight_gap  daily_return_pct
count       60600.00       60600.00  ...       60600.00          60600.00
mean           -0.01           0.01  ...           0.03              0.06
std             1.01           1.04  ...           1.19              1.96
min            -6.88          -3.12  ...         -29.66            -35.12
25%            -0.57          -0.62  ...          -0.39             -0.86
50%            -0.01          -0.23  ...           0.04              0.06
75%             0.56           0.34  ...           0.46              0.99
max             6.69           7.48  ...          37.52             24.50

[8 rows x 5 columns]


In [0]:
# Train Isolation Forest
iso_forest = IsolationForest(n_estimators=200, contamination=0.03, random_state=42)

X_anomaly = anomaly_df[feature_cols_anomaly].values  # .values removes column names warning

# FIT the model first
iso_forest.fit(X_anomaly)

# Now predict
anomaly_df["anomaly_score"] = iso_forest.decision_function(X_anomaly)
anomaly_df["is_anomaly"] = iso_forest.predict(X_anomaly)

# Isolation Forest outputs: 1 = normal, -1 = anomaly
anomaly_df["is_anomaly"] = (anomaly_df["is_anomaly"] == -1).astype(int)

total_anomalies = anomaly_df["is_anomaly"].sum()
print(f"Total trading days: {len(anomaly_df)}")
print(f"Anomalies detected: {total_anomalies} ({total_anomalies/len(anomaly_df)*100:.1f}%)")
print()

# Show the most extreme anomalies
print("=" * 60)
print("TOP 20 MOST EXTREME ANOMALIES")
print("=" * 60)
top_anomalies = anomaly_df[anomaly_df["is_anomaly"] == 1].nsmallest(20, "anomaly_score")
print(top_anomalies[["Date", "Ticker", "Sector", "Close", "daily_return_pct", 
                       "return_zscore", "volume_zscore", "anomaly_score"]].to_string(index=False))
print()

# Anomalies by sector
print("=" * 60)
print("ANOMALIES BY SECTOR")
print("=" * 60)
sector_anomalies = anomaly_df.groupby("Sector").agg(
    total_days=("is_anomaly", "count"),
    anomaly_days=("is_anomaly", "sum")
).reset_index()
sector_anomalies["anomaly_pct"] = (sector_anomalies["anomaly_days"] / sector_anomalies["total_days"] * 100).round(2)
print(sector_anomalies.sort_values("anomaly_pct", ascending=False).to_string(index=False))

Total trading days: 60600
Anomalies detected: 1818 (3.0%)

TOP 20 MOST EXTREME ANOMALIES
      Date Ticker      Sector      Close  daily_return_pct  return_zscore  volume_zscore  anomaly_score
2025-10-06    AMD  Technology 203.710007           23.7080       5.805052       5.758938      -0.208485
2024-02-02   META  Technology 471.291534           20.3176       6.692150       6.838939      -0.205479
2023-08-08    LLY  Healthcare 510.601562           14.8696       6.436744       6.881708      -0.203029
2023-05-25   NVDA  Technology  37.946404           24.3696       6.285114       5.830978      -0.200314
2023-02-02   META  Technology 187.300156           23.2824       5.380774       5.031723      -0.195182
2022-02-01    UPS Industrials 185.930740           14.0844       6.199560       6.949389      -0.194375
2022-04-20   NFLX       Media  22.618999          -35.1166      -6.071180       7.317586      -0.194106
2024-02-08    DIS       Media 108.348717           11.4989       5.636351      

In [0]:
# Save anomaly results to Databricks
anomaly_results = anomaly_df[["Date", "Ticker", "Sector", "Close", "Volume",
                               "daily_return_pct", "return_zscore", "volume_zscore",
                               "range_zscore", "overnight_gap", "anomaly_score", 
                               "is_anomaly"]].copy()

spark_anomaly = spark.createDataFrame(anomaly_results)
spark_anomaly.write.mode("overwrite").saveAsTable("gold_anomaly_detection")

print(f"Saved {len(anomaly_results)} rows to gold_anomaly_detection")
print(f"  Normal days: {len(anomaly_results[anomaly_results['is_anomaly']==0])}")
print(f"  Anomaly days: {len(anomaly_results[anomaly_results['is_anomaly']==1])}")

# Also save sector anomaly summary
sector_summary = spark.createDataFrame(sector_anomalies)
sector_summary.write.mode("overwrite").saveAsTable("gold_anomaly_by_sector")

print(f"\nSector summary saved to gold_anomaly_by_sector")

Saved 60600 rows to gold_anomaly_detection
  Normal days: 58782
  Anomaly days: 1818

Sector summary saved to gold_anomaly_by_sector
